<a href="https://colab.research.google.com/github/knah1d/unsharp-image_processing/blob/main/endoscopy_srgan_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write('KGAT_933919b12637cc381b805c623cbc9007')
os.system('chmod 600 /root/.kaggle/access_token')

0

In [ ]:
!pip install -q kaggle
!kaggle datasets list -s kvasir

ref                                                               title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
meetnagadia/kvasir-dataset                                        Kvasir Dataset                                       1238130953  2022-03-17 07:38:46.120000           5498         54  0.875            
plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset            Kvasir v2                                            2493169587  2021-04-26 09:05:27.047000           2670         26  0.875            
yasserhessein/the-kvasir-dataset                                  The Kvasir Dataset                                   2493213587  2021-10-10 17:45:29.963000           3036         50  0.6

In [ ]:
DATA_DIR = '/content/drive/MyDrive/endoscopy_srgan/data'
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d meetnagadia/kvasir-dataset -p {DATA_DIR}/kvasir --unzip
!kaggle datasets download -d balraj98/cvcclinicdb -p {DATA_DIR}/cvc_clinicdb --unzip
!kaggle datasets download -d nguyenvoquocduong/etis-laribpolypdb -p {DATA_DIR}/etis_larib --unzip

Dataset URL: https://www.kaggle.com/datasets/meetnagadia/kvasir-dataset
License(s): ODbL-1.0
100% 1.15G/1.15G [01:07<00:00, 18.3MB/s]

Dataset URL: https://www.kaggle.com/datasets/balraj98/cvcclinicdb
License(s): other
100% 131M/131M [00:07<00:00, 17.7MB/s]

Dataset URL: https://www.kaggle.com/datasets/nguyenvoquocduong/etis-laribpolypdb
License(s): unknown
100% 177M/177M [00:10<00:00, 18.1MB/s]



In [ ]:
for d in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, d)
    print(d, '->', len(os.listdir(path)), 'items:', os.listdir(path)[:10])

kvasir -> 1 items: ['kvasir-dataset']
cvc_clinicdb -> 4 items: ['PNG', 'TIF', 'class_dict.csv', 'metadata.csv']
etis_larib -> 2 items: ['images', 'masks']


In [ ]:
!pip install -q -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.8/256.8 kB 22.2 MB/s eta 0:00:00


In [ ]:
import os

def peek(path, depth=2):
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/  ({len(files)} files)')
        for f in files[:3]:
            print(f'{indent}  {f}')

peek(os.path.join(DATA_DIR, 'kvasir'))
print('---')
peek(os.path.join(DATA_DIR, 'cvc_clinicdb'))
print('---')
peek(os.path.join(DATA_DIR, 'etis_larib'))

kvasir/  (0 files)
  kvasir-dataset/  (0 files)
    dyed-lifted-polyps/  (500 files)
      0053d7cd-549c-48cd-b370-b4ad64a8098a.jpg
      007d5aa7-7289-4bad-aa4a-5c3a259e9b19.jpg
      00cf9508-6ad1-4db9-840a-519c1d515c30.jpg
    dyed-resection-margins/  (500 files)
      009bd044-4a5a-4c81-b481-d20fb223cd81.jpg
      00adb051-3e76-4482-b0e5-5207a028470b.jpg
      01086b46-817d-4f20-8aba-4a7b9347ab1c.jpg
    esophagitis/  (500 files)
      001fb927-4814-4ba5-851d-189db99291d8.jpg
      00687a70-bbad-4bf9-864f-9f7b3c27a2c8.jpg
      0134d93d-0922-4063-9acd-a4177f2b0c07.jpg
    normal-cecum/  (500 files)
      00b99f19-2c31-4c7c-931a-4c3b38d70d1a.jpg
      00f3d2cc-93ea-40f0-9b88-b159b07a49cb.jpg
      0163b3a2-9aa1-4b23-a9a8-aff07a738f16.jpg
    normal-pylorus/  (500 files)
      005959d0-b75b-41ed-8da1-2a5d0666d612.jpg
      026715f1-d01e-47c7-ae95-93b46695a7e8.jpg
      02e55fb2-8a74-4671-95a9-81a97e9a6210.jpg
    normal-z-line/  (500 files)
      00bee375-36d2-4ba9-89e5-bd6132d79c0c.

In [ ]:
import glob, random, json

paths = []
paths += glob.glob(os.path.join(DATA_DIR, 'kvasir/kvasir-dataset/*/*.jpg'))
paths += glob.glob(os.path.join(DATA_DIR, 'cvc_clinicdb/PNG/Original/*.png'))
paths += glob.glob(os.path.join(DATA_DIR, 'etis_larib/images/*.png'))

print('total images found:', len(paths))

random.seed(42)
random.shuffle(paths)
n_val = int(0.1 * len(paths))
val_paths, train_paths = paths[:n_val], paths[n_val:]

manifest = {'train': train_paths, 'val': val_paths}
manifest_path = os.path.join(DATA_DIR, 'manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f)

print('train:', len(train_paths), '| val:', len(val_paths))
print('saved manifest to', manifest_path)

total images found: 4808
train: 4328 | val: 480
saved manifest to /content/drive/MyDrive/endoscopy_srgan/data/manifest.json


In [ ]:
!git clone https://github.com/knah1d/unsharp-image_processing.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt 2>/dev/null || pip install -q opencv-python-headless torch torchvision scikit-image

Cloning into '/content/repo'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 53 (delta 0), reused 53 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 20.80 MiB | 17.79 MiB/s, done.
/content/repo


In [ ]:
!pip install -q opencv-python-headless torch torchvision scikit-image lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.9 MB/s eta 0:00:00


In [ ]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.28 KiB | 656.00 KiB/s, done.
From https://github.com/knah1d/unsharp-image_processing
   0e5ce07..af07a43  main       -> origin/main
Updating 0e5ce07..af07a43
Fast-forward
 enhance.py     | 36 +++++++++++++++++++++++++++++++++++-
 train_srgan.py | 23 ++++++++++++++++++++++-
 2 files changed, 57 insertions(+), 2 deletions(-)


In [ ]:
!python train_srgan.py \
    --manifest /content/drive/MyDrive/endoscopy_srgan/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_test \
    --epochs 2 --pretrain_epochs 1 --batch_size 8

[train_srgan] device = cuda
[train_srgan] train=4328  val=480
[epoch 0] (pretrain) D=0.0000  G=0.0574
[epoch 1] (adversarial) D=0.2060  G=0.0454
[epoch 1] val_mse=0.003427  val_psnr=24.65dB
[train_srgan] done. Final generator weights: /content/drive/MyDrive/endoscopy_srgan/checkpoints_test/srgan_v.pth


In [ ]:
%cd /content/repo
!git pull

/content/repo
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 2.25 KiB | 2.25 MiB/s, done.
From https://github.com/knah1d/unsharp-image_processing
   af07a43..fda5d88  main       -> origin/main
Updating af07a43..fda5d88
Fast-forward
 train_srgan.py | 131 +++++++++++++++++++++++++++++++++++++++++++++++++--------
 1 file changed, 113 insertions(+), 18 deletions(-)


In [ ]:
%cd /content/repo
!git pull
!python train_srgan.py \
    --manifest /content/drive/MyDrive/endoscopy_srgan/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_test3 \
    --epochs 3 --pretrain_epochs 1 --batch_size 8

/content/repo
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 4), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 2.97 KiB | 1013.00 KiB/s, done.
From https://github.com/knah1d/unsharp-image_processing
   fda5d88..daeffd3  main       -> origin/main
Updating fda5d88..daeffd3
Fast-forward
 enhance.py     |  22 ++++++++++-
 train_srgan.py | 123 ++++++++++++++++++++++++++++++++++++++++++++++++++-------
 2 files changed, 129 insertions(+), 16 deletions(-)
[train_srgan] device = cuda
[train_srgan] train=4328  val=480
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100% 548M/548M [00:06<00:00, 89.0MB/s]
/content/repo/train_srgan.py:410: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `o

In [ ]:

%cd /content/repo
!git pull
!python train_srgan.py \
    --manifest /content/drive/MyDrive/endoscopy_srgan/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_test4 \
    --epochs 3 --pretrain_epochs 1 --batch_size 8

/content/repo
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 533 bytes | 533.00 KiB/s, done.
From https://github.com/knah1d/unsharp-image_processing
   daeffd3..9b1168d  main       -> origin/main
Updating daeffd3..9b1168d
Fast-forward
 train_srgan.py | 16 ++++++++++------
 1 file changed, 10 insertions(+), 6 deletions(-)
[train_srgan] device = cuda
[train_srgan] train=4328  val=480
/content/repo/train_srgan.py:410: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched_

In [ ]:
!python train_srgan.py \
    --manifest /content/drive/MyDrive/endoscopy_srgan/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints \
    --epochs 50 --pretrain_epochs 5 --batch_size 16

[train_srgan] device = cuda
[train_srgan] train=4328  val=480
Traceback (most recent call last):
  File "/content/repo/train_srgan.py", line 480, in <module>
    train(parse_args())
  File "/content/repo/train_srgan.py", line 319, in train
    sched_G.load_state_dict(ckpt["sched_G"])
                            ~~~~^^^^^^^^^^^
KeyError: 'sched_G'


In [ ]:
!rm -f /content/drive/MyDrive/endoscopy_srgan/checkpoints/srgan_last.pth
!rm -f /content/drive/MyDrive/endoscopy_srgan/checkpoints/srgan_v.pth

In [ ]:
!python train_srgan.py \
    --manifest /content/drive/MyDrive/endoscopy_srgan/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints \
    --epochs 50 --pretrain_epochs 5 --batch_size 16

[train_srgan] device = cuda
[train_srgan] train=4328  val=480
/content/repo/train_srgan.py:410: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched_D.step()
[epoch 0] (pretrain) D=0.0000  G=0.1105  lr=1.00e-04
           raw terms -> pixel=0.0554  content=0.8215  adv=0.0000  hpf=0.0140
[epoch 1] (pretrain) D=0.0000  G=0.0940  lr=1.00e-04
           raw terms -> pixel=0.0463  content=0.7066  adv=0.0000  hpf=0.0124
[epoch 2] (pretrain) D=0.0000  G=0.0908  lr=1.00e-04
           raw terms -> pixel=0.0446  content=0.6825  adv=0.0000  hpf=0.0121
[epoch 3] (pretrain) D=0.0000  G=0.0859  lr=1.00e-04
           raw terms -> pixel=0.0409  content=0.6630  

In [ ]:
%cd /content/repo
!git pull
!rm -f /content/drive/MyDrive/endoscopy_srgan/checkpoints/srgan_last.pth
!rm -f /content/drive/MyDrive/endoscopy_srgan/checkpoints/srgan_v.pth
!python train_srgan.py \
    --manifest /content/drive/MyDrive/endoscopy_srgan/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_test5 \
    --epochs 3 --pretrain_epochs 1 --batch_size 8

/content/repo
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 4.91 MiB | 9.57 MiB/s, done.
From https://github.com/knah1d/unsharp-image_processing
   9b1168d..ddc03c3  main       -> origin/main
Updating 9b1168d..ddc03c3
Fast-forward
 .gitignore     |   3 ++-
 srgan_v.pth    | Bin 0 -> 5601859 bytes
 train_srgan.py |  17 +++++++++++++----
 3 files changed, 15 insertions(+), 5 deletions(-)
 create mode 100644 srgan_v.pth
[train_srgan] device = cuda
[train_srgan] train=4328  val=480
/content/repo/train_srgan.py:419: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. 

In [ ]:
!ls -la /content/drive/MyDrive/endoscopy_srgan/checkpoints/

total 0


In [ ]:
!python train_srgan.py \
    --manifest /content/drive/MyDrive/endoscopy_srgan/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints \
    --epochs 50 --pretrain_epochs 5 --batch_size 16

[train_srgan] device = cuda
[train_srgan] train=4328  val=480
/content/repo/train_srgan.py:419: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched_D.step()
[epoch 0] (pretrain) D=0.0000  G=0.1267  lr=1.00e-04
           raw terms -> pixel=0.0554  content=0.9125  adv=0.0000  hpf=0.0256
[epoch 1] (pretrain) D=0.0000  G=0.1108  lr=1.00e-04
           raw terms -> pixel=0.0468  content=0.8068  adv=0.0000  hpf=0.0236
[epoch 2] (pretrain) D=0.0000  G=0.1071  lr=1.00e-04
           raw terms -> pixel=0.0446  content=0.7852  adv=0.0000  hpf=0.0233
[epoch 3] (pretrain) D=0.0000  G=0.1049  lr=1.00e-04
           raw terms -> pixel=0.0432  content=0.7719  